# 中国数据中心投资量化模型 - 端到端 Demo

> 这个 Notebook 演示从数据加载 → 综合评分 → IRR 计算 → 敏感性分析 → 可视化的完整流程。
> 面试时建议用这个 Notebook 现场展示（10 分钟版本）。

**作者**: Sky Shen  
**背景**: 经济学本科 + 房地产/基础设施硕士 + Data Science 硕士 (在读) + CFA L1 (在考)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

## 1. 加载数据 (开箱即用)

`data/reference/` 中已经整理好了 5 张参考表，无需 API 调用即可开始分析。

In [ ]:
from src.model.site_score import load_all_data

data = load_all_data()
print('已加载数据集:')
for k, df in data.items():
    print(f'  {k:12s}: {len(df)} 行')

### 1.1 行业背景：A 股 / 美股 IDC 上市公司概览

In [ ]:
from src.data_collection.listed_company_idcs import get_market_overview

overview = get_market_overview()
overview.sort_values('market_cap_cny_bn', ascending=False).head(10)

### 1.2 数据中心专项电价最低 Top 5

**核心洞察**：电费占 IDC 现金 OPEX 的 55-70%，是选址第一变量。

In [ ]:
from src.data_collection.electricity_prices import find_cheapest_provinces
find_cheapest_provinces(top_n=8)

## 2. 单城市财务模型：以贵阳为例

假设新建 1000 机柜的 AI 数据中心，单机柜 8kW，PUE 1.25。

In [ ]:
from src.model.irr import calculate_idc_irr

result = calculate_idc_irr(
    city='贵阳',
    rack_count=1000,
    rack_power_kw=8.0,
    pue=1.25,
    electricity_price=0.35,         # 贵州数据中心专项价
    rack_monthly_revenue=3500,
    capex_per_rack=80000,
    ramp_curve='conservative',
    project_years=10,
    discount_rate=0.08,
)
print(f'城市: {result["city"]}')
print(f'IRR: {result["irr"]:.2%}')
print(f'NPV @ 8%: ¥{result["npv"]/1e6:.1f}M')
print(f'稳定期 Cap Rate: {result["cap_rate_stabilized"]:.2%}')
print(f'回收期: {result["payback_years"]} 年')
print(f'总 CAPEX: ¥{result["total_capex"]/1e6:.1f}M')

In [ ]:
cf = result['cashflow_df'].copy()
for col in ['revenue', 'cash_opex', 'ebitda', 'depreciation', 'net_income', 'fcf']:
    cf[col] = (cf[col] / 1e6).round(2)
cf[['year', 'revenue', 'cash_opex', 'ebitda', 'depreciation', 'net_income', 'fcf']]

## 3. 25 城市综合选址评分

In [ ]:
from src.model.site_score import score_cities

scored = score_cities(data['cities'], data['power'], data['climate'], data['nodes'])

cols = ['city', 'province', 'tier', 'power_price', 'policy_score',
        'cooling_score', 'composite_score']
scored[cols].head(15)

## 4. 25 城市 IRR 比较（核心结论）

In [ ]:
from src.data_collection.climate_data import estimate_pue_from_climate

results = []
for _, row in scored.iterrows():
    # 用各城市实际参数：电价、气候PUE、机柜租金
    pue = estimate_pue_from_climate(
        data['climate'].set_index('city').loc[row['city'], 'avg_annual_temp_c']
        if row['city'] in data['climate']['city'].values else 15
    )
    r = calculate_idc_irr(
        city=row['city'],
        rack_count=1000,
        rack_power_kw=8.0,
        pue=pue,
        electricity_price=row['power_price'],
        rack_monthly_revenue=row['monthly_rack_rent_yuan'],
        capex_per_rack=80000,
        ramp_curve='aggressive' if row['tier'] == 'T1' else ('standard' if row['tier'] == 'T2' else 'conservative'),
    )
    results.append({
        'city': row['city'],
        'tier': row['tier'],
        'composite_score': row['composite_score'],
        'irr': r['irr'],
        'npv_mn': r['npv'] / 1e6,
        'cap_rate': r['cap_rate_stabilized'],
        'payback_yrs': r['payback_years'],
    })

results_df = pd.DataFrame(results).sort_values('irr', ascending=False)
results_df['irr_display'] = (results_df['irr'] * 100).round(2).astype(str) + '%'
results_df['cap_rate_display'] = (results_df['cap_rate'] * 100).round(2).astype(str) + '%'
results_df.head(10)[['city', 'tier', 'composite_score', 'irr_display', 'cap_rate_display', 'npv_mn', 'payback_yrs']]

**核心结论**：
- IRR 最高的几乎都是 **东数西算西部节点 + 低电价** 的组合
- 一线城市虽然租金高，但电价 + 土地成本 + 能耗约束让 IRR 反而较低
- **政策红利是真实的财务影响**，不只是行政符号

## 5. 敏感性分析：IRR 对电价的弹性

In [ ]:
base = {
    'city': '贵阳',
    'rack_count': 1000,
    'rack_power_kw': 8.0,
    'pue': 1.25,
    'electricity_price': 0.35,
    'rack_monthly_revenue': 3500,
    'capex_per_rack': 80000,
    'ramp_curve': 'conservative',
}

prices = np.arange(0.25, 0.75, 0.05)
irrs = []
for p in prices:
    params = base.copy()
    params['electricity_price'] = p
    r = calculate_idc_irr(**params)
    irrs.append(r['irr'] * 100)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(prices, irrs, marker='o', linewidth=2, color='#1f4e78')
ax.axhline(y=8, color='red', linestyle='--', alpha=0.5, label='Hurdle Rate 8%')
ax.set_xlabel('电价 (元/kWh)')
ax.set_ylabel('IRR (%)')
ax.set_title('贵阳项目 IRR 对电价的敏感性')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

elasticity = (irrs[-1] - irrs[0]) / (prices[-1] - prices[0])
print(f'\n敏感度: 电价每变化 0.01 元/kWh, IRR 变化约 {elasticity * 0.01:.2f} pp')

## 6. 可视化：全国 IDC 投资机会地图

In [ ]:
from src.visualization.china_map import plot_idc_opportunity_map
from pathlib import Path

out = Path('../outputs/china_idc_opportunity_map.html')
plot_idc_opportunity_map(scored, out)
print(f'✓ 交互地图已生成: {out.resolve()}')
print('  在浏览器中打开即可交互式查看每个候选城市的评分明细')

## 7. 投资建议（一句话）

基于模型：

1. **首选标的**: 贵安新区、中卫、和林格尔（IRR 最高，政策+电价+气候三重红利）
2. **次选**: 张家口、韶关、芜湖（承接东部需求的枢纽内集群）
3. **机会型**: 乌兰察布、庆阳（新兴节点，先发优势）
4. **回避**: 一线城市新建（能耗约束 + 高电价 + 高地价，IRR < 8%）

**对应上市公司机会**：
- **GDS 万国数据 / 润泽科技** — 已重仓京津冀 + 廊坊，吃东部需求
- **世纪互联 / 数据港** — 中卫 + 张北布局完整
- **奥飞数据** — 粤港澳延伸（韶关）
- **宝信软件** — 工业 IDC + 国资背景，有政策红利

## 8. 模型局限（面试主动讲）

1. **客户分布未建模**：实际 AI 训练客户分布在京沪深，西部需要专线连接，时延和带宽成本未量化
2. **电价假设静态**：未来 3-5 年新能源占比上升可能进一步压低绿电价格
3. **碳配额未考虑**：东部省份 PUE 强制要求会进一步逼高一线城市成本
4. **现金流简化**：未拆分建设期 1-2 年 CAPEX 分期投入
5. **入住率曲线**：用行业基准而非客户合同管线建模

**下一版改进**：
- 引入客户管线建模（基于公开新闻 + LLM 文本分析）
- 加入 REITs 退出估值（Cap Rate × 稳定期 NOI）做 Total Return
- Monte Carlo 模拟电价/入住率不确定性下的 IRR 分布